In [6]:
import pickle
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from ta.momentum import RSIIndicator, ROCIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands, AverageTrueRange

In [3]:
with open("model_cat.pkl", "rb") as file:
    model_cat = pickle.load(file)

print("CatBoost model loaded successfully!")

CatBoost model loaded successfully!


In [4]:
# Load the original dataset
data = pd.read_csv("merged_data.csv")

# Convert Date to datetime
data["Date"] = pd.to_datetime(data["Date"])

print("Dataset shape:", data.shape)
print("Date range:", data["Date"].min(), "to", data["Date"].max())

Dataset shape: (1381571, 7)
Date range: 2000-01-01 00:00:00 to 2025-02-26 00:00:00


## **Feature Engineering**

In [8]:
from ta.momentum import RSIIndicator, ROCIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands, AverageTrueRange

# Sort data
data = data.sort_values(
    ["Scrip", "Date"]
).reset_index(drop=True)


def add_features(group):

    close = group["Close"]
    high = group["High"]
    low = group["Low"]
    volume = group["Volume"]

    # -----------------------------
    # Basic Features
    # -----------------------------

    group["Return"] = close.pct_change()

    group["Return_Lag1"] = group["Return"].shift(1)
    group["Return_Lag2"] = group["Return"].shift(2)
    group["Return_Lag5"] = group["Return"].shift(5)

    group["Volatility_5"] = (
        group["Return"].rolling(5).std()
    )

    group["Volatility_20"] = (
        group["Return"].rolling(20).std()
    )

    group["Price_Range"] = (
        (high - low) / close
    )

    group["Volume_Change"] = volume.pct_change()

    group["Volume_SMA_20"] = (
        volume.rolling(20).mean()
    )


    # -----------------------------
    # RSI
    # -----------------------------

    if len(group) >= 14:

        rsi = RSIIndicator(
            close=close,
            window=14
        )

        group["RSI"] = rsi.rsi()

    else:

        group["RSI"] = np.nan


    # -----------------------------
    # MACD
    # -----------------------------

    if len(group) >= 26:

        macd = MACD(
            close=close,
            window_slow=26,
            window_fast=12,
            window_sign=9
        )

        group["MACD"] = macd.macd()
        group["MACD_Signal"] = macd.macd_signal()
        group["MACD_Hist"] = macd.macd_diff()

    else:

        group["MACD"] = np.nan
        group["MACD_Signal"] = np.nan
        group["MACD_Hist"] = np.nan


    # -----------------------------
    # Bollinger Band
    # -----------------------------

    if len(group) >= 20:

        bb = BollingerBands(
            close=close,
            window=20
        )

        group["BB_High"] = bb.bollinger_hband()

    else:

        group["BB_High"] = np.nan


    # -----------------------------
    # ATR
    # -----------------------------

    if len(group) >= 14:

        atr = AverageTrueRange(
            high=high,
            low=low,
            close=close,
            window=14
        )

        group["ATR"] = atr.average_true_range()

    else:

        group["ATR"] = np.nan


    # -----------------------------
    # ROC
    # -----------------------------

    if len(group) >= 12:

        roc = ROCIndicator(
            close=close,
            window=12
        )

        group["ROC"] = roc.roc()

    else:

        group["ROC"] = np.nan


    return group


# Apply feature engineering
data = data.groupby(
    "Scrip",
    group_keys=False
).apply(add_features)

data = data.reset_index(drop=True)

print("Feature engineering completed!")
print("Shape:", data.shape)

Feature engineering completed!
Shape: (1381571, 23)


In [9]:
FEATURES = [
    "Open",
    "High",
    "Volume",
    "Return",
    "Return_Lag1",
    "Return_Lag2",
    "Return_Lag5",
    "Volatility_5",
    "Volatility_20",
    "Price_Range",
    "Volume_Change",
    "Volume_SMA_20",
    "RSI",
    "MACD",
    "MACD_Signal",
    "MACD_Hist",
    "BB_High",
    "ATR",
    "ROC"
]

X_all = data[FEATURES].copy()

# Replace infinity
X_all = X_all.replace(
    [np.inf, -np.inf],
    np.nan
)

# Fill missing values
X_all = X_all.fillna(
    X_all.median()
)

print("Feature matrix shape:", X_all.shape)
print("Number of features:", len(FEATURES))
print("Remaining NaN:", X_all.isna().sum().sum())

Feature matrix shape: (1381571, 19)
Number of features: 19
Remaining NaN: 0
